In [1]:
import numpy as np
import pandas as pd
import pickle
import os 
import json 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
RANDOM = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

RAW_DIR = "../data/raw"
PROCESS_DIR = "../data/processed"
SPLIT_DIR = f"{PROCESS_DIR}/split"

os.makedirs(SPLIT_DIR, exist_ok=True)

CLASS_NAMES = {
    0: "Benign",
    1: "Analysis",
    2: "Backdoor",
    3: "DoS",
    4: "Exploits",
    5: "Fuzzers",
    6: "Generic",
    7: "Reconnaissance",
    8: "Shellcode",
    9: "Worms",
}

In [4]:
X = pd.read_csv(f"{RAW_DIR}/Dataset.csv")
y = pd.read_csv(f"{RAW_DIR}/Label.csv")

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (447915, 76), y shape: (447915, 1)


In [5]:
features_names = list(X.columns)
n_features = len(features_names)
print(n_features)

76


In [6]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = TEST_SIZE, stratify=y, random_state=RANDOM)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=VAL_SIZE / ( 1 - TEST_SIZE), stratify=y_temp,random_state=RANDOM)


In [9]:
total = len(X_train) + len(X_val) + len(X_test)

for name, y_s in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} set distribution:")
    print(y_s.value_counts(normalize=True))

Train set distribution:
Label
0        0.800003
4        0.069098
5        0.066113
7        0.037364
6        0.010340
3        0.009973
8        0.004695
2        0.001008
1        0.000858
9        0.000549
Name: proportion, dtype: float64
Val set distribution:
Label
0        0.799994
4        0.069105
5        0.066113
7        0.037358
6        0.010344
3        0.009972
8        0.004688
2        0.001012
1        0.000863
9        0.000551
Name: proportion, dtype: float64
Test set distribution:
Label
0        0.799994
4        0.069105
5        0.066113
7        0.037358
6        0.010344
3        0.009972
8        0.004688
2        0.001012
1        0.000863
9        0.000551
Name: proportion, dtype: float64


In [ ]:
# Split only for XGBOOST , before scale
np.save(f"{SPLIT_DIR}/X_train.npy", X_train.values)
np.save(f"{SPLIT_DIR}/y_train.npy", y_train.values)
np.save(f"{SPLIT_DIR}/X_val.npy", X_val.values)
np.save(f"{SPLIT_DIR}/y_val.npy", y_val.values)
np.save(f"{SPLIT_DIR}/X_test.npy", X_test.values)
np.save(f"{SPLIT_DIR}/y_test.npy", y_test.values)

In [ ]:
# Scale
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train.values)
X_val_scale = scaler.transform(X_val.values)
X_test_scale = scaler.transform(X_test.values)

mean_check = np.allclose(X_train_scale.mean(axis=0), 0, atol=1e-7)
std_check = np.allclose(X_train_scale.std(axis=0), 1, atol=1e-7)
print(f"train mean : {mean_check}, train std : {std_check}")
print(f"scaler mean : {scaler.mean_}, scaler var : {scaler.var_}")

np.save(f"{SPLIT_DIR}/X_train_scale.npy", X_train_scale)
np.save(f"{SPLIT_DIR}/X_val_scale.npy", X_val_scale)
np.save(f"{SPLIT_DIR}/X_test_scale.npy", X_test_scale)


train mean : True, train std : False
scaler mean : [6.00644570e+05 2.27409318e+01 2.72298853e+01 4.89967912e+03
 2.16253237e+04 1.73884496e+02 1.48438025e+01 7.09703152e+01
 6.38228201e+01 3.66004283e+02 6.68152287e+00 1.73207225e+02
 1.51511431e+02 2.45449444e+06 1.61805664e+04 2.39983375e+04
 5.33779381e+04 2.23060506e+05 1.15030819e+02 5.75644423e+05
 3.39481101e+04 6.87126516e+04 2.25375905e+05 1.27539598e+02
 4.36357171e+05 3.40936313e+04 6.55351092e+04 2.08113611e+05
 4.63371702e+00 7.23992869e-04 0.00000000e+00 0.00000000e+00
 0.00000000e+00 6.67830464e+02 8.25542736e+02 1.34512435e+04
 2.72932289e+03 1.21105955e+01 4.71263074e+02 1.39103405e+02
 1.79055155e+02 9.47484797e+04 9.37762128e-01 1.80635902e+00
 9.56818769e-05 1.75522758e+01 4.82021567e+01 0.00000000e+00
 0.00000000e+00 0.00000000e+00 1.09599763e+00 1.47053419e+02
 7.09703152e+01 1.73207225e+02 0.00000000e+00 0.00000000e+00
 0.00000000e+00 2.21415322e+04 2.54929467e+01 6.09151771e+05
 3.17506594e+00 9.29629226e+02 4.9

In [14]:
with open(f"{PROCESS_DIR}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print(f"X_train_scale shape: {X_train_scale.shape}, X_val_scale shape: {X_val_scale.shape}, X_test_scale shape: {X_test_scale.shape}    ")

X_train_scale shape: (313539, 76), X_val_scale shape: (67188, 76), X_test_scale shape: (67188, 76)    


In [17]:
metadata = {
    "n_samples" : total,
    "n_features" : n_features,
    "n_classes" : len(CLASS_NAMES),
    "label_mapping" : {v: k for k, v in CLASS_NAMES.items()},
    "features" : features_names,
    "split" : {
        "train" : {"n" : len(X_train), "pct" : len(X_train) / total},
        "val" : {"n" : len(X_val), "pct" : len(X_val) / total},
        "test" : {"n" : len(X_test), "pct" : len(X_test) / total}
    },
    "scaler"  :{
        "type" : "StandardScaler",
        "n_features" : len(scaler.mean_),
        "mean_min" : float(scaler.mean_.min()),
        "mean_max" : float(scaler.mean_.max()),
    }, 
    "random_state" : RANDOM
}

with open(f"{SPLIT_DIR}/split_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"class : {list(CLASS_NAMES.values())}")
print(f"features : {n_features}")

class : ['Benign', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
features : 76
